# 06 — Corrected primary-results analysis

This notebook analyzes the completed 108-run primary sweep. It does not train models or include the exploratory hybrid work. Results are validation-set estimates because the same validation split was used in learning-rate selection.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from this notebook directory or from the repository root.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
RESULTS_DIR = REPO_ROOT / 'results'
RAW_RESULTS = RESULTS_DIR / '05-full-experiment-sweep' / 'experiment_results.csv'
ANALYSIS_DIR = RESULTS_DIR / '06-results-analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_RESULTS)
expected_runs = 3 * 2 * 6 * 3
assert len(df) == expected_runs, f'Expected {expected_runs} runs, found {len(df)}'
assert df[['method', 'language', 'budget', 'seed']].duplicated().sum() == 0
print(f'Loaded {len(df)} primary validation runs from {RAW_RESULTS}')

In [ ]:
# A collapsed run predicts at (or extremely close to) the one-class accuracy baseline.
# With three balanced labels, a constant-class predictor has macro-F1 near 1/6.
collapsed = df[np.isclose(df['accuracy'], 1 / 3, atol=0.001)].copy()
collapsed.to_csv(ANALYSIS_DIR / 'collapsed_runs.csv', index=False)
print(f'Collapsed runs: {len(collapsed)} / {len(df)} ({100 * len(collapsed) / len(df):.1f}%)')
display(collapsed.groupby('method').size().rename('collapsed_runs'))
display(collapsed.groupby(['method', 'budget']).size().rename('collapsed_runs'))

In [ ]:
# Correct 95% intervals for n=3 seeds use t_(0.975, df=2) = 4.302652729.
# These intervals describe uncertainty across seeds; they are not pairwise significance tests.
T_CRITICAL_DF2 = 4.302652729
summary = df.groupby(['method', 'language', 'budget']).agg(
    f1_mean=('macro_f1', 'mean'),
    f1_std=('macro_f1', 'std'),
    acc_mean=('accuracy', 'mean'),
    acc_std=('accuracy', 'std'),
    n=('seed', 'count'),
).reset_index()
assert (summary['n'] == 3).all()
summary['f1_ci95'] = T_CRITICAL_DF2 * summary['f1_std'].fillna(0) / np.sqrt(summary['n'])
summary['acc_ci95'] = T_CRITICAL_DF2 * summary['acc_std'].fillna(0) / np.sqrt(summary['n'])
summary['ci_method'] = 'two-sided t interval; df=2; t*=4.302652729'
summary = summary.sort_values(['language', 'method', 'budget'])
summary.to_csv(ANALYSIS_DIR / 'summary_with_ci.csv', index=False)
display(summary)

In [ ]:
METHODS = ['lora', 'dora', 'ia3']
BUDGETS = [50, 100, 500, 1000, 2000, 20000]
COLORS = {'lora': '#1f77b4', 'dora': '#ff7f0e', 'ia3': '#2ca02c'}

ranking_table = summary.pivot(index=['language', 'budget'], columns='method', values='f1_mean')
ranking_table = ranking_table[METHODS]
ranking_table['highest_mean_method'] = ranking_table.idxmax(axis=1)
ranking_table.to_csv(ANALYSIS_DIR / 'ranking_table.csv')
display(ranking_table)

comparison = ranking_table.reset_index().pivot(index='budget', columns='language', values='highest_mean_method')
agreement = (comparison['hi'] == comparison['te']).mean()
print(f'Highest-mean-method agreement across languages: {agreement:.1%}')
display(comparison)

In [ ]:
# Record rank-order changes only. A change is not labelled a statistical crossover.
rank_changes = []
for language in ['hi', 'te']:
    language_rows = summary[summary['language'] == language]
    previous_ranking = None
    previous_budget = None
    for budget in BUDGETS:
        ranking = language_rows[language_rows['budget'] == budget].sort_values('f1_mean', ascending=False)['method'].tolist()
        if previous_ranking is not None and ranking != previous_ranking:
            rank_changes.append({
                'language': language,
                'budget_before': previous_budget,
                'budget_after': budget,
                'ranking_before': ' > '.join(previous_ranking),
                'ranking_after': ' > '.join(ranking),
            })
        previous_ranking, previous_budget = ranking, budget
rank_changes = pd.DataFrame(rank_changes)
rank_changes.to_csv(ANALYSIS_DIR / 'rank_order_changes.csv', index=False)
display(rank_changes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
for ax, language, title in zip(axes, ['hi', 'te'], ['Hindi', 'Telugu']):
    for method in METHODS:
        sub = summary[(summary['method'] == method) & (summary['language'] == language)].sort_values('budget')
        ax.plot(sub['budget'], sub['f1_mean'], marker='o', label=method.upper(), color=COLORS[method])
        ax.fill_between(sub['budget'], sub['f1_mean'] - sub['f1_ci95'], sub['f1_mean'] + sub['f1_ci95'], alpha=0.15, color=COLORS[method])
    ax.set_xscale('log')
    ax.set_xlabel('Training budget (samples, log scale)')
    ax.set_title(f'{title} — validation macro-F1')
    ax.axhline(1 / 3, color='gray', linestyle='--', linewidth=1, label='Expected random-guess macro-F1')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Macro-F1')
axes[0].legend(loc='lower right', fontsize=9)
fig.suptitle('Mean validation macro-F1 with 95% t intervals (n=3 seeds)', y=1.02)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# This memory field is measured by the training notebook after a no-gradient forward pass.
# Do not describe it as peak training memory.
compute_summary = df.groupby('method').agg(
    trainable_params=('trainable_params', 'first'),
    avg_forward_pass_memory_gb=('peak_gpu_memory_gb', 'mean'),
    total_training_time_sec=('training_time_sec', 'sum'),
).reset_index()
compute_summary['total_training_time_min'] = compute_summary['total_training_time_sec'] / 60
compute_summary.to_csv(ANALYSIS_DIR / 'compute_efficiency.csv', index=False)
display(compute_summary)

## Reporting guidance

Report these as validation-set means with three-seed t intervals. Do not make test-set, statistical-significance, or confirmed-crossover claims from this notebook. Use a separate held-out test evaluation before finalizing the study.